# 01 - 全体成績サマリー

メンバー全員のフレックスランク成績を俯瞰する。

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns
import yaml
from pathlib import Path

for font in ['Meiryo', 'Yu Gothic', 'MS Gothic', 'Hiragino Sans']:
    try:
        matplotlib.font_manager.findfont(font, fallback_to_default=False)
        plt.rcParams['font.family'] = font
        break
    except ValueError:
        continue
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid', font=plt.rcParams['font.family'])

DATA = Path('..') / 'data' / 'processed'
CONFIG = Path('..') / 'config' / 'settings.yaml'

with open(CONFIG, encoding='utf-8') as f:
    cfg = yaml.safe_load(f)
MEMBERS = {f'{m["game_name"]}#{m["tag_line"]}' for m in cfg['members']}

df_matches = pd.read_csv(DATA / 'matches.csv', parse_dates=['gameCreation'])
df_players = pd.read_csv(DATA / 'player_stats.csv', parse_dates=['gameCreation'])

df_members = df_players[
    df_players.apply(lambda r: f'{r["summonerName"]}#{r["tagLine"]}' in MEMBERS, axis=1)
].copy()

print(f'総試合数: {len(df_matches)}')
print(f'メンバーの延べ参加レコード数: {len(df_members)}')

In [ ]:
# メンバー別 勝率・試合数・平均KDA
summary = df_members.groupby('summonerName').agg(
    試合数=('win', 'count'),
    勝利数=('win', 'sum'),
    勝率=('win', 'mean'),
    平均KDA=('kda', 'mean'),
    平均キル=('kills', 'mean'),
    平均デス=('deaths', 'mean'),
    平均アシスト=('assists', 'mean'),
).round(2)
summary['勝率'] = (summary['勝率'] * 100).round(1).astype(str) + '%'
summary.sort_values('勝利数', ascending=False)

In [ ]:
# 直近20試合の勝率推移（メンバー別）
fig, ax = plt.subplots(figsize=(12, 5))

for name, grp in df_members.sort_values('gameCreation').groupby('summonerName'):
    rolling_wr = grp['win'].astype(int).rolling(window=20, min_periods=5).mean() * 100
    ax.plot(range(len(rolling_wr)), rolling_wr, label=name, linewidth=2)

ax.set_xlabel('試合 (古い → 新しい)')
ax.set_ylabel('勝率 % (直近20試合移動平均)')
ax.set_title('メンバー別 勝率推移')
ax.legend()
ax.axhline(y=50, color='gray', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# メンバー別 ロール分布
role_order = ['TOP', 'JUNGLE', 'MIDDLE', 'BOTTOM', 'UTILITY']
role_counts = df_members.groupby(['summonerName', 'role']).size().unstack(fill_value=0)
role_counts = role_counts.reindex(columns=[r for r in role_order if r in role_counts.columns])

role_counts.plot(kind='bar', stacked=True, figsize=(10, 5), colormap='Set2')
plt.title('メンバー別 ロール分布')
plt.xlabel('')
plt.ylabel('試合数')
plt.legend(title='ロール')
plt.tight_layout()
plt.show()